# Composite score

Aggregates attribution recall, attribution precision, and ensemble stability into a single quantifiable attribution benchmark metric. 

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
tpr_results_path = "./experiments/dnn_spikein/tpr_results.csv"
ppv_results_path = "./experiments/dnn_decoy/ppv_results.csv"
ens_results_path = "./experiments/ensemble_10m_3L/ens_stability.csv"

recall_df = pd.read_csv(tpr_results_path)
recall_df.rename(columns={'algo': 'algorithm'}, inplace=True)
recall_df.drop(index=recall_df[recall_df['algorithm']=='gwas'].index.values, inplace=True)
precision_df = pd.read_csv(ppv_results_path)
precision_df.rename(columns={'algo': 'algorithm'}, inplace=True)
ens_df = pd.read_csv(ens_results_path)
ens_df.rename(columns={'Algorithm': 'algorithm'}, inplace=True)
recall_df.rename(columns={'algo': 'algorithm'}, inplace=True)
precision_df.rename(columns={'algo': 'algorithm'}, inplace=True)

for algo in np.unique(recall_df['algorithm'].values):
    if algo not in precision_df['algorithm'].values:
        print(f"Algorithm '{algo}' found in recall but not precision.")
    if algo not in ens_df['algorithm'].values:
        print(f"Algorithm '{algo}' found in recall but not stability.")

for algo in np.unique(precision_df['algorithm'].values):
    if algo not in recall_df['algorithm'].values:
        print(f"Algorithm '{algo}' found in precision but not recall.")
    if algo not in ens_df['algorithm'].values:
        print(f"Algorithm '{algo}' found in precision but not stability.")

for algo in ens_df['algorithm'].values:
    if algo not in recall_df['algorithm'].values:
        print(f"Algorithm '{algo}' found in stability but not recall.")
    if algo not in precision_df['algorithm'].values:
        print(f"Algorithm '{algo}' found in stability but not precision.")

In [ ]:
# get recall and precision at a specific quantile threshold
q_threshold = 0.98
recall_at_q_df = recall_df.loc[np.isclose(recall_df['quantile'].values, q_threshold, atol=1e-6)]
precision_at_q_df = precision_df.loc[np.isclose(precision_df['quantile'].values, q_threshold, atol=1e-6)]
# df --> series
recall_at_q = recall_at_q_df.set_index('algorithm').loc[:,'syn_tpr']
precision_at_q = precision_at_q_df.set_index('algorithm').loc[:,'ppv']
print(f"Recall at q={q_threshold}:\n{recall_at_q}\n")
print(f"Precision at q={q_threshold}:\n{precision_at_q}\n")

In [ ]:
def transform_stability_half_life(stability_series, tau=None):
    """
    Convert (in)stability measures (i.e., median RSD across 
    ensemble members for each algorithm) into a stability 
    score in (0,1] using half-life decay: S = 2 ** (-x / tau)

    where x is the (in)stability or median RSD (higher = worse), and
    tau is half-life; the instability level that halves the score.

    Args:
    stability_series: pd.Series
        Raw (in)stability for each algorithm (e.g. median RSD across SNPs).
        Index should be algorithm names.
    tau: float or None
        Half-life parameter. If None, we set tau = stability_series.median().

    Returns: pd.Series
        Stability scores in (0,1], higher = better (more stable).
    """
    if tau is None:
        tau = stability_series.median()
    if tau == 0:
        tau = 1e-12
    stability_score = 2 ** (-stability_series / tau)
    return pd.Series(stability_score, index=stability_series.index)

ens_hl_norm = transform_stability_half_life(ens_df.set_index('algorithm').loc[:,'median'], tau=None)

In [ ]:
composite_df = pd.merge(recall_at_q, precision_at_q, left_index=True, right_index=True)
composite_df = pd.merge(composite_df, ens_hl_norm, left_index=True, right_index=True)

# Compute composite score (primary=geometric mean;
# also included is arithmetic and harmonic means for comparison)
composite_df['composite_arithmetic'] = (composite_df['syn_tpr'] + composite_df['ppv'] + composite_df['median']) / 3
composite_df['composite_harmonic'] = 3 / (1/composite_df['syn_tpr'] + 1/composite_df['ppv'] + 1/composite_df['median'])
composite_df['composite_geometric'] = (composite_df['syn_tpr'] * composite_df['ppv'] * composite_df['median']) ** (1/3)

composite_file = "./composite_results.csv"
composite_df.to_csv(composite_file, index=True, index_label='algorithm')

In [ ]:
composite_df.sort_values(by='composite_geometric', ascending=False, inplace=True)